#week11 / MNIST



컴퓨터공학과 / 202433638 / 장영환

주교재 376p 예제 : MNIST 필기체 숫자 인식

CNN 모델 구성 → 컴파일 → 학습

CNN의 Conv2D는 입력을 다음 형태로 받는다.

(데이터 개수, 높이, 너비, 채널) , MNIST는 흑백 이미지라서 채널이 1개

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

# MNIST 데이터셋 불러오기
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

# CNN 입력 형태로 변환: (데이터 개수, 높이, 너비, 채널)
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# 픽셀 값을 0~1 사이로 정규화
train_images, test_images = train_images / 255.0, test_images / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


MNIST도 픽셀 값이 원래 0~255이다.

이를 0~1 사이 값으로 바꾼다.

In [ ]:
model = models.Sequential()

# 1번째 합성곱 층
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))

# 1번째 풀링 층
model.add(layers.MaxPooling2D((2, 2)))

# 2번째 합성곱 층
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# 2번째 풀링 층
model.add(layers.MaxPooling2D((2, 2)))

# 3번째 합성곱 층
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# 2차원 특징맵을 1차원 벡터로 변환
model.add(layers.Flatten())

# 완전연결층
model.add(layers.Dense(64, activation='relu'))

# 출력층: MNIST는 0~9 숫자 분류이므로 클래스 10개
model.add(layers.Dense(10, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,322 (364.54 KB)

 Trainable params: 93,322 (364.54 KB)

 Non-trainable params: 0 (0.00 B)

optimizer는 모델의 가중치를 어떻게 수정할지 정하는 방법

예측
->
정답과 비교
->
loss 계산
->
가중치 수정
->
다시 예측

이때 가중치 수정 방법이 optimizer이다.

loss는 모델이 얼마나 틀렸는지 나타내는 값이다.

MNIST에서는 다음 조건 때문에 이 손실 함수를 쓴다.

다중 분류 문제

정답 라벨이 정수 형태

출력층이 softmax

MNIST 라벨은 원-핫 인코딩이 아니다.
정수 라벨을 그대로 쓰기 때문에 sparse_categorical_crossentropy를 사용

In [ ]:
model.compile(optimizer = 'adam',
loss = 'sparse_categorical_crossentropy',
metrics =['accuracy'])
model.fit(train_images, train_labels, epochs = 5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 31ms/step - accuracy: 0.9550 - loss: 0.1454
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 57s 30ms/step - accuracy: 0.9860 - loss: 0.0462
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9895 - loss: 0.0333
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 30ms/step - accuracy: 0.9923 - loss: 0.0254
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9934 - loss: 0.0202
